In [1]:
from abc import ABC, abstractmethod
import random

class Agent(ABC):
    """
    Abstract Base Class for all market participants.
    Enforces that every agent must have a 'get_action' method.
    """
    def __init__(self, agent_id, initial_cash, initial_inventory):
        self.agent_id = agent_id
        self.cash = initial_cash
        self.inventory = initial_inventory
        
    @abstractmethod
    def get_action(self, market_snapshot):
        """
        Input: 
            market_snapshot (dict): Contains 'best_bid', 'best_ask', 'last_price', 'time'
        
        Output: 
            Order dictionary or None
            Format: {'side': 'buy'|'sell', 'price': float, 'qty': int, 'type': 'limit'|'market'}
        """
        pass

class RandomAgent(Agent):
    """
    A 'Zero Intelligence' agent for testing.
    It buys or sells randomly, ignoring price (Noise Trader).
    """
    def __init__(self, agent_id, cash, inventory, arrival_rate=1.0):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate # Probability of acting per tick

    def get_action(self, market_snapshot):
        # 1. Poisson Arrival Check (Simulate random arrival times)
        if random.random() > self.arrival_rate:
            return None 

        # 2. Random Direction (50/50)
        side = 'buy' if random.random() > 0.5 else 'sell'
        
        # 3. Determine Price
        # To ensure execution in our simulation, we place aggressive Limit orders
        # (Buy slightly above ask, Sell slightly below bid)
        best_bid = market_snapshot.get('best_bid')
        best_ask = market_snapshot.get('best_ask')
        mid_price = market_snapshot.get('mid_price')
        
        # Fallback if book is empty
        ref_price = mid_price if mid_price else 100.0
        
        if side == 'buy':
            # Aggressive buy: Price = Best Ask + Noise (or Ref + Noise)
            base = best_ask if best_ask else ref_price
            price = round(base + random.uniform(0, 2), 2)
        else:
            # Aggressive sell: Price = Best Bid - Noise (or Ref - Noise)
            base = best_bid if best_bid else ref_price
            price = round(base - random.uniform(0, 2), 2)
            
        qty = random.randint(1, 10)
        
        # 4. Construct Order
        return {
            'agent_id': self.agent_id,
            'side': side,
            'price': price,
            'qty': qty,
            'type': 'limit'
        }

In [20]:
def test_random_agent():
    print("--- Testing Random Agent ---")
    
    # 1. Initialize Agent
    trader_bob = RandomAgent(agent_id=1, cash=10000, inventory=0)
    
    # 2. Create a Mock Market Snapshot (What the agent sees)
    fake_market = {
        'best_bid': 99.0,
        'best_ask': 101.0,
        'mid_price': 100.0,
        'time': '09:30:00'
    }
    
    # 3. Ask Agent for an Action
    action = trader_bob.get_action(fake_market)
    
    # 4. Validate Output
    if action:
        print(f"Agent Action: {action['side'].upper()} {action['qty']} @ {action['price']}")
        
        # Validation Logic
        if action['side'] == 'buy':
            assert action['price'] >= 101.0, "Buy price should be aggressive (>= Best Ask)"
        else:
            assert action['price'] <= 99.0, "Sell price should be aggressive (<= Best Bid)"
            
        print("✅ Logic Check Passed")
    else:
        print("Agent decided not to trade (Arrival check failed).")

if __name__ == "__main__":
    test_random_agent()

--- Testing Random Agent ---
Agent Action: SELL 5 @ 97.12
✅ Logic Check Passed
